# HAFTA 5

Bu hafta öğreneceklerin:

- Her ara değişkenin gradient'ini elle yazmak ve PyTorch'unkiyle karşılaştırmak
- Broadcasting'in gradient'e etkisi: toplanan boyut geri dönerken nerede sum alınır
- Cross entropy ve BatchNorm'un türevlerinin neden tek satıra indiği

In [29]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [30]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [31]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [32]:
# build the dataset
block_size = 3 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [33]:
# utility function we will use later when comparing manual gradients to PyTorch gradients
def cmp(s, dt, t):
  ex = torch.all(dt == t.grad).item()
  app = torch.allclose(dt, t.grad)
  maxdiff = (dt - t.grad).abs().max().item()
  print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [34]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 64 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1 # using b1 just for fun, it's useless because of BN
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

# Note: I am initializating many of these parameters in non-standard ways
# because sometimes initializating with e.g. all zeros could mask an incorrect
# implementation of the backward pass.

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

4137


In [35]:
batch_size = 32
n = batch_size # a shorter variable also, for convenience
# construct a minibatch
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

### Görev 1
Geçen haftaki MLP + BatchNorm modelini videodaki gibi küçük adımlara böl (logits, counts, probs, logprobs, ...) ve loss.backward() ile her ara değişkenin gradient'ini al. (Egzersiz 1, videonun ilk yarısı)

Backpropu otomatikleştirmiştik en son.
İlk haftalardan hatırlamamız gereken Backprop kuralları:

1. Boyut Eşitliği Kuralı: 
    - Bir tensörün türevi (gradyanı), o tensörün kendisiyle birebir aynı boyutta (shape) olmak zorundadır. Örneğin W2 boyutu (64, 27) ise dW2 de kesinlikle (64, 27) olmalıdır.

2. Zincir Kuralı (Chain Rule): Bir girdi için gradyan:$$\text{d(girdi)} = \text{(lokal türev)} \times \text{d(çıktı)}$$

3. Broadcasting - Summing İkiliği (Duality):

    - İleri yayılımda (forward pass) küçük bir tensör kopyalanarak genişletilmişse (broadcasting), geriye yayılımda (backward pass) o eksen boyunca toplam (sum) alınmalıdır.
    - İleri yayılımda bir eksen toplanarak küçültülmüşse (sum), geriye yayılımda o eksene gradyan kopyalanarak (broadcast/replicate) dağıtılır.

4. Dallanma (Branching) Kuralı: 
    -  Eğer bir değişken ileri yayılımda birden fazla yerde kullanıldıysa, geriye yayılımda o değişkene gelen tüm gradyanlar toplanır (+=).

Şimdi normalde otomatikleştirerel loss.backward() ile backpropagation uyguladığımız koda ve çıktımıza göz atalım.

Ve çıktılara göre nasıl türev almamız gerektiğinin mantığını kavrayalım manuel hesaplamak için.

In [36]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time

emb = C[Xb] # embed the characters into vectors
embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
# Linear layer 1
hprebn = embcat @ W1 + b1 # hidden layer pre-activation
# BatchNorm layer
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
# Non-linearity
h = torch.tanh(hpreact) # hidden layer
# Linear layer 2
logits = h @ W2 + b2 # output layer
# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean() # BURADAN BAŞLIYORUZ

# PyTorch backward pass
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad()
loss.backward()
loss

tensor(3.3620, grad_fn=<NegBackward0>)

### Doğrulama (`cmp` Fonksiyonu) Nasıl Yorumlanır?

Her bir ara tensör türevini yazdığında Karpathy'nin hazırladığı karşılaştırma fonksiyonunu çalıştıracağız:

```python cmp('logprobs', dlogprobs, logprobs) cmp('probs', dprobs, probs) ...

```

- ****EXACT** (True):** Bit düzeyinde PyTorch'un `autograd`'ı ile birebir aynı sonucu buldun.
- ****APPROXIMATE** (True):** Kayan noktalı sayı (floating point) işlem sırasından kaynaklı $10^{-8}$ gibi minik yuvarlama farkları var ama matematiksel olarak tamamen doğru.
- ****EXACT** False & **APPROX** False:** Türevde veya tensörün toplanma boyutunda (`sum(0)` vs `sum(1)`) bir hata var demektir.

---

#### 1. `loss` $\to$ `dlogprobs`

- **İleri Yayılım:** Doğru harflerin log olasılıkları çekilir ve mini-batch ortalaması alınıp eksi ile çarpılır:

$$\text{loss} = -\frac{1}{N} \sum_{i=1}^N \text{logprobs}[i, y_i]$$

- **Mantık & Türev:**
- `loss` ifadesinin ortalamadaki tek bir doğru harfe göre lokal türevi $-\frac{1}{N}$'dir ($N = 32$).
- Yanlış sınıfların `loss` üzerinde hiçbir etkisi olmadığı için onların gradyanı $0$'dır.

- **Nasıl Kurgulanır?**
`logprobs` ile aynı boyutta `(32, 27)` sıfırlardan oluşan bir tensör açılır (`torch.zeros_like(logprobs)`). Yalnızca doğru sınıf indekslerine ($i, y_i$) $-1.0 / N$ atanır.

---

In [37]:
# loss = -logprobs[range(n), Yb].mean() # türevi alıncak kod

dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0 / n

cmp('logprobs', dlogprobs, logprobs)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0


#### 2. `dlogprobs` $\to$ `dprobs`

- **İleri Yayılım:** `logprobs = probs.log()`
- **Mantık & Türev:**
- $\frac{d}{dx} \ln(x) = \frac{1}{x}$.
- Lokal türev $\frac{1}{\text{probs}}$ olur.

- **Nasıl Kurgulanır?**
Zincir kuralı gereği: `dprobs = (1.0 / probs) * dlogprobs`.
*(Sezgi: Eğer model doğru sınıfa çok düşük olasılık vermişse, $1 / \text{probs}$ devasa bir sayı olur ve o hatanın gradyanını büyüterek modeli sertçe uyarır).*

---

In [38]:
# logprobs = probs.log() # türevi alıncak kod
dprobs = (1.0 / probs) * dlogprobs

cmp('probs', dprobs, probs)

probs           | exact: True  | approximate: True  | maxdiff: 0.0


#### 3. `dprobs` $\to$ `dcounts_sum_inv` ve `dcounts` (1. Kol)

- **İleri Yayılım:** `probs = counts * counts_sum_inv`
- `counts`: `(32, 27)`
- `counts_sum_inv`: `(32, 1)` (Burada 27 sütun boyunca broadcast edilmiştir).

- **Mantık & Türev:** $C = A \cdot B$ çarpımında:
- $\frac{\partial C}{\partial B} = A$ ve $\frac{\partial C}{\partial A} = B$.

- **Nasıl Kurgulanır?**
- `dcounts`: Lokal türev `counts_sum_inv`'dir. Zincir kuralıyla: `counts_sum_inv * dprobs` (boyutlar `(32, 27)` olarak doğrudan eşleşir).
- `dcounts_sum_inv`: Lokal türev `counts`'tur (`counts * dprobs`). Ancak `counts_sum_inv` `(32, 1)` boyutundadır ve 27 sütuna yayılmıştır. **Kural 3** gereği: 27 sütun boyunca toplanmalıdır (`.sum(1, keepdim=True)`).

---

In [39]:
# probs = counts * counts_sum_inv # türevi alıncak kod

dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
dcounts = counts_sum_inv * dprobs

cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts', dcounts, counts)

# NOT 1:
# Bu kısımda counts değeri false çıkması gerekli 
# Sebebi 4. kuraldaki dallanma kuralından dolayıdır. 
# counts_sum_inv ve counts değişkenleri aynı değişkene bağlıdır, yani counts değişkeni 2 kez kullanırız
# Bu yüzden counts değişkeni false çıkmaktadır.
# pytorchun counts.grad değeri bu 2 kolu da hesaplıyor ve toplamını alıyor. 
# Bizim manuel türevimiz ise şuanda sadece 1 kolu hesaplıyor.

# NOT 2:
# Burada cmp('counts', dcounts, counts) çağrılırsa FALSE verir!
# ÇÜNKÜ: 'counts' tensörü ileri yayılımda iki yerde kullanılmıştır (Dallanma / Branching):
#   1. Kol: probs = counts * counts_sum_inv (şu an hesapladığımız)
#   2. Kol: counts_sum = counts.sum(1, keepdims=True) (henüz hesaplamadığımız)
# 4. Kural gereği, 2. kolun gradyanı da hesaplanıp 'dcounts' üzerine eklenmeden (+=)
# PyTorch'un counts.grad değerine eşit olamaz. Test 5. adımda yapılacaktır.

counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: False | approximate: False | maxdiff: 0.005767475813627243


#### 4. `dcounts_sum_inv` $\to$ `dcounts_sum`

- **İleri Yayılım:** `counts_sum_inv = counts_sum**(-1)`
- **Mantık & Türev:**
- $\frac{d}{dx}(x^{-1}) = -x^{-2} = -\frac{1}{x^2}$.

- **Nasıl Kurgulanır?**
Lokal türev $-(counts\_sum)^{-2}$ olur.
Zincir kuralı: `-(counts_sum**(-2)) * dcounts_sum_inv`.

---

In [40]:
# counts_sum_inv = counts_sum**-1 # türevi alıncak kod

dcounts_sum = -(counts_sum**-2) * dcounts_sum_inv
cmp('counts_sum', dcounts_sum, counts_sum)

counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0


#### 5. `dcounts_sum` $\to$ `dcounts` (2. Kol ve Birleştirme)

- **İleri Yayılım:** `counts_sum = counts.sum(1, keepdim=True)`
- **Mantık & Türev:**
- İleri yayılımda sütunlar toplanarak `(32, 1)` haline getirilmiştir.
- Bir toplama düğümü gradyan dağıtıcıdır (router). Gelen gradyan, o toplama katılan 27 sütunun her birine aynen kopyalanır (**Kural 3**).

- **Nasıl Kurgulanır?**
`dcounts_sum` sütun tensörü 27 sütun boyunca tekrarlanır (örneğin `torch.ones_like(counts) * dcounts_sum`).
**Kural 4:** `counts` hem `probs` hesabında hem de `counts_sum` hesabında kullanıldığı için iki koldan gelen gradyan toplanmalıdır: `dcounts += ...`

---

In [41]:
# counts_sum = counts.sum(1, keepdims=True) # türevi alıncak kod

# İleri yayılımda yatay toplam (sum(1)) yapıldığı için geriye yayılımda gradyan
# 27 sütun boyunca broadcast edilir (router mantığı).
# Kural 4 branching: İki koldan gelen gradyanlar toplanır (+=).
dcounts += torch.ones_like(counts) * dcounts_sum

# ARTIK HER İKİ KOL DA TAMAMLANDI, TEST ŞİMDİ DOĞRULANABİLİR
cmp('counts', dcounts, counts)


counts          | exact: True  | approximate: True  | maxdiff: 0.0


#### 6. `dcounts` $\to$ `dnorm_logits`

- **İleri Yayılım:** `counts = norm_logits.exp()`
- **Mantık & Türev:**
- $\frac{d}{dx} e^x = e^x$.
- Lokal türev doğrudan $e^{\text{norm\_logits}}$, yani zaten elimizde olan `counts` tensörüdür.

- **Nasıl Kurgulanır?**
`dnorm_logits = counts * dcounts`.

---


In [42]:
# counts = norm_logits.exp() # türevi alıncak kod

dnorm_logits = counts * dcounts
cmp('norm_logits', dnorm_logits, norm_logits)

norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0


#### 7. `dnorm_logits` $\to$ `dlogits` (1. Kol) ve `dlogit_maxes`

- **İleri Yayılım:** `norm_logits = logits - logit_maxes`
- `logit_maxes`: `(32, 1)` (27 sütuna broadcast edilmiştir).

- **Mantık & Türev:**
- `logits`'e göre lokal türev $+1$'dir.
- `logit_maxes`'a göre lokal türev $-1$'dir.

- **Nasıl Kurgulanır?**
- `dlogits = dnorm_logits.clone()` (ilk kol).
- `dlogit_maxes`: Lokal türev $-1$ olduğundan `-dnorm_logits` alınır ve broadcast edilen 1. boyut (27 sütun) boyunca toplanır: `(-dnorm_logits).sum(1, keepdim=True)`.

---

In [43]:
# norm_logits = logits - logit_maxes # türevi alıncak kod

dlogits = dnorm_logits.clone()
dlogit_maxes = (-dnorm_logits).sum(1, keepdim=True)

cmp('logit_maxes', dlogit_maxes, logit_maxes)

logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0


#### 8. `dlogit_maxes` $\to$ `dlogits` (2. Kol)

- **İleri Yayılım:** `logit_maxes = logits.max(1, keepdim=True).values`
- **Mantık & Türev:**
- Her satırda yalnızca maksimum olan tek bir eleman seçilmiştir.
- Gradyan sadece o maksimumun geldiği sütuna akar (lokal türevi 1), diğer 26 sütuna $0$ akar.

- **Nasıl Kurgulanır?**
Maksimum indekslerin konumunda $1$, diğer yerlerde $0$ olan bir one-hot maskesi oluşturulur (`F.one_hot(indices, num_classes=27)`). Bu maske `dlogit_maxes` ile çarpılarak `dlogits`'e eklenir: `dlogits += ...`

---

In [44]:
# logit_maxes = logits.max(1, keepdim=True).values # türevi alıncak kod

dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes

cmp('logits', dlogits, logits)

logits          | exact: True  | approximate: True  | maxdiff: 0.0


#### 9. Doğrusal Katman 2 (`logits` $\to$ `dh`, `dW2`, `db2`)

- **İleri Yayılım:** `logits = h @ W2 + b2`
- `h`: `(32, 64)`, `W2`: `(64, 27)`, `b2`: `(27,)`, `logits`: `(32, 27)`

- **Mantık & Boyut Eşleme:**
Matris çarpımı türevinde formül ezberlemek yerine **Boyut Kuralı (Kural 1)** uygulanır:
- `dh` boyutu `(32, 64)` olmalı $\to$ `dlogits (32, 27)` ile `W2.T (27, 64)` çarpılmalı: `dlogits @ W2.T`.
- `dW2` boyutu `(64, 27)` olmalı $\to$ `h.T (64, 32)` ile `dlogits (32, 27)` çarpılmalı: `h.T @ dlogits`.
- `db2` boyutu `(27,)` olmalı $\to$ `b2` satırlar boyunca broadcast edildiği için 0. boyut boyunca toplanmalı: `dlogits.sum(0)`.

---

In [45]:
# logits = h @ W2 + b2 # türevi alıncak kod

# Mantık: Matris çarpımının türevi yine bir matris çarpımıdır.
# Karpathy'nin belirttiği gibi formül ezberlemek yerine tensör boyutları (shape matching)
# takip edilir:
#   - dh  için hedef (32, 64) -> dlogits (32, 27) @ W2.T (27, 64)
#   - dW2 için hedef (64, 27) -> h.T (64, 32) @ dlogits (32, 27)
#   - db2 için hedef (27,)   -> dlogits 0. eksen (batch) boyunca toplanır.

dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)

cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)

# videoda 53. dakika civari benzer sonuçlar
# NOT: 'exact: False' ancak 'approximate: True' çıkmasının sebebi:
# Donanım seviyesinde matris çarpımı (BLAS) yapılırken kayan noktalı sayıların 
# (floating point) paralel toplama sıralamasından kaynaklanan ~1e-8 mertebesindeki 
# mikroskobik yuvarlama farkıdır. Matematiksel ve tensör boyutu açısından türev tamamen doğrudur.

# bu şekilde yazdığımızda false ve approximate değerleri True çıktı bende



h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0


#### 10. Aktivasyon Katmanı (`dh` $\to$ `dhpreact`)

- **İleri Yayılım:** `h = torch.tanh(hpreact)`
- **Mantık & Türev:**
- $\frac{d}{dx} \tanh(x) = 1 - \tanh^2(x) = 1 - h^2$.

- **Nasıl Kurgulanır?**
`dhpreact = (1.0 - h**2) * dh`.

---

In [46]:
# h = torch.tanh(hpreact) # hidden layer # türevi alıncak kod

dhpreact = (1.0 - h**2) * dh
cmp('hpreact', dhpreact, hpreact)

# NOT: exact: False, approximate: True çıkması beklenen durumdur.
# Sebebi: 9. adımdaki dh tensöründen zincirleme olarak devralınan 
# ~2e-9 mertebesindeki float32 yuvarlama farkıdır.
# Türev matematiksel ve algoritmik olarak doğrudur.

hpreact         | exact: True  | approximate: True  | maxdiff: 0.0


#### 11. BatchNorm Ölçek ve Kaydırma (`dhpreact` $\to$ `dbngain`, `dbnbias`, `dbnraw`)

- **İleri Yayılım:** `hpreact = bngain * bnraw + bnbias`
- `bngain`, `bnbias`: `(1, 64)`
- `bnraw`, `hpreact`: `(32, 64)`

- **Nasıl Kurgulanır?**
- `dbnraw`: Lokal türev `bngain` $\to$ `bngain * dhpreact`.
- `dbngain`: Lokal türev `bnraw` $\to$ `(bnraw * dhpreact).sum(0, keepdim=True)` (32 örnek boyunca toplanır).
- `dbnbias`: Lokal türev $+1$ $\to$ `dhpreact.sum(0, keepdim=True)`.

---

In [47]:
# hpreact = bngain * bnraw + bnbias # BatchNorm layer # türevi alıncak kod

dbnraw = bngain * dhpreact
dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
dbnbias = dhpreact.sum(0, keepdim=True)

cmp('bnraw', dbnraw, bnraw)
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)

# NOT: exact: False, approximate: True çıkması beklenen durumdur.
# Sebebi: 9. adımdaki dhpreact tensöründen zincirleme

bnraw           | exact: True  | approximate: True  | maxdiff: 0.0
bngain          | exact: True  | approximate: True  | maxdiff: 0.0
bnbias          | exact: True  | approximate: True  | maxdiff: 0.0


#### 12. BatchNorm Normalizasyon Bölümü (`dbnraw` $\to$ `dbndiff` [1. Kol] ve `dbnvar_inv`)

- **İleri Yayılım:** `bnraw = bndiff * bnvar_inv`
- `bndiff`: `(32, 64)`
- `bnvar_inv`: `(1, 64)`

- **Nasıl Kurgulanır?**
- `dbndiff = bnvar_inv * dbnraw` (1. kol; 2. kol daha sonra varyans üzerinden gelecek!).
- `dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)`.

---

In [48]:
# bnraw = bndiff * bnvar_inv # BatchNorm layer # türevi alıncak kod
dbndiff = bnvar_inv * dbnraw
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)

cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bndiff', dbndiff, bndiff)

# NOT: bndiff burada False çıkar 
# çünkü 2. kol henüz bndiff2 ve bnvar üzerinden geri dönmemiştir. 
# 2. kol hesaplanıp (15. adım) dbndiff += ... yapıldığında True olacaktır.

bnvar_inv       | exact: True  | approximate: True  | maxdiff: 0.0
bndiff          | exact: False | approximate: False | maxdiff: 0.001199284684844315


#### 13. `dbnvar_inv` $\to$ `dbnvar` (Tam 1:05:00 civarı)

- **İleri Yayılım:** `bnvar_inv = (bnvar + 1e-5)**(-0.5)`
- **Mantık & Türev:**
- $\frac{d}{dx}(x^{-0.5}) = -0.5 \cdot x^{-1.5} = -0.5 \cdot (x + \epsilon)^{-1.5}$.

- **Nasıl Kurgulanır?**
`dbnvar = (-0.5 * (bnvar + 1e-5)**(-1.5)) * dbnvar_inv`.

---

In [49]:
# bnvar_inv = (bnvar + 1e-5)**-0.5 # BatchNorm layer # türevi alıncak kod

dbnvar = (-0.5 * (bnvar + 1e-5)**(-1.5)) * dbnvar_inv

cmp('bnvar', dbnvar, bnvar)

bnvar           | exact: True  | approximate: True  | maxdiff: 0.0


#### 14. `dbnvar` $\to$ `dbndiff2`

- **İleri Yayılım:** `bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True)`
*(Not: Bessel düzeltmesi nedeniyle $n-1$'e bölünmüştür. Boyutlar: `bndiff2` $\in \mathbb{R}^{32 \times 64}$, `bnvar` $\in \mathbb{R}^{1 \times 64}$).*

- **Mantık & Türev:**
- İleri yayılımda 0. boyut (32 satır) boyunca toplanan tensör, geriye yayılımda **Kural 3 (Broadcasting - Summing İkiliği)** gereği tüm satırlara kopyalanır (broadcast).

- Lokal türev $\frac{1}{n-1}$ katsayısıdır.

- **Nasıl Kurgulanır?**
`dbndiff2 = (1.0 / (n - 1)) * torch.ones_like(bndiff2) * dbnvar`.

---


In [50]:
# bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # BatchNorm layer # türevi alıncak kod

dbndiff2 = (1.0 / (n - 1)) * torch.ones_like(bndiff2) * dbnvar

cmp('bndiff2', dbndiff2, bndiff2)

bndiff2         | exact: True  | approximate: True  | maxdiff: 0.0


#### 15. `dbndiff2` $\to$ `dbndiff` (2. Kol ve Birleştirme)

- **İleri Yayılım:** `bndiff2 = bndiff**2`
- **Mantık & Türev:**
- $\frac{d}{dx}(x^2) = 2x$ olduğundan lokal türev $2 \cdot \text{bndiff}$'tir.

- **Kural 4 (Dallanma / Branching):** `bndiff`, hem `bnraw` (Adım 12) hem de `bndiff2` hesabında kullanılmıştır. 12. adımda hesaplanan ilk kolun üzerine bu ikinci kol eklenir (`+=`):

$$\left. \frac{\partial \text{loss}}{\partial \text{bndiff}} \right\vert{}_{\text{toplam}} = \left. \frac{\partial \text{loss}}{\partial \text{bndiff}} \right\vert{}_{\text{kol 1}} + (2 \cdot \text{bndiff}) \odot \text{dbndiff2}$$

- **Nasıl Kurgulanır?**
`dbndiff += (2.0 * bndiff) * dbndiff2`.

---

In [51]:
# bndiff2 = bndiff**2 # BatchNorm layer # türevi alıncak kod

# 2. koldan gelen gradyanı mevcut dbndiff üzerine ekliyoruz (Branching kuralı):
dbndiff += (2.0 * bndiff) * dbndiff2

# ARTIK HER İKİ KOL DA TAMAMLANDI, TEST ŞİMDİ DOĞRULANABİLİR
cmp('bndiff', dbndiff, bndiff)

bndiff          | exact: True  | approximate: True  | maxdiff: 0.0


#### 16. `dbndiff` $\to$ `dhprebn` (1. Kol) ve `dbnmeani`

- **İleri Yayılım:** `bndiff = hprebn - bnmeani`
(Not: `bnmeani` $\in \mathbb{R}^{1 \times 64}$, 32 satır boyunca broadcast edilmiştir).

- **Mantık & Türev:**
- `hprebn`'e göre lokal türev $+1$'dir (1. kol).

- `bnmeani`'ye göre lokal türev $-1$'dir ve broadcast edilen 0. boyut (32 satır) boyunca toplanmalıdır.

- **Nasıl Kurgulanır?**
- `dhprebn = dbndiff.clone()` (ilk kol).

- `dbnmeani = (-dbndiff).sum(0, keepdim=True)`.

---


In [52]:
# bndiff = hprebn - bnmeani # BatchNorm layer # türevi alıncak kod

dhprebn = dbndiff.clone()
dbnmeani = (-dbndiff).sum(0, keepdim=True)

cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)

# NOT: dhprebn burada False çıkar çünkü 2. kol henüz bnmeani üzerinden geri dönmemiştir.
# 2. kol hesaplanıp (17. adım) dhprebn += ... yapıldığında True olacaktır.

bnmeani         | exact: True  | approximate: True  | maxdiff: 0.0
hprebn          | exact: False | approximate: False | maxdiff: 0.0011352961882948875


#### 17. `dbnmeani` $\to$ `dhprebn` (2. Kol ve Birleştirme)

- **İleri Yayılım:** `bnmeani = 1/n*hprebn.sum(0, keepdim=True)`

- **Mantık & Türev:**
- İleri yayılımda ortalama almak için 0. boyut toplanmıştır. Geriye yayılımda gradyan tüm satırlara broadcast edilir ve $\frac{1}{N}$ ile ölçeklenir.

- **Kural 4 (Dallanma):** İki koldan gelen gradyanlar toplanır (`+=`):

$$\left. \frac{\partial \text{loss}}{\partial \text{hprebn}} \right\vert{}_{\text{toplam}} = \left. \frac{\partial \text{loss}}{\partial \text{hprebn}} \right\vert{}_{\text{kol 1}} + \frac{1}{N} \cdot \frac{\partial \text{loss}}{\partial \text{bnmeani}}$$

- **Nasıl Kurgulanır?**
`dhprebn += (1.0 / n) * torch.ones_like(hprebn) * dbnmeani`.

---

In [53]:
# bnmeani = 1/n*hprebn.sum(0, keepdim=True) # BatchNorm layer # türevi alıncak kod

# İleri yayılımda dikey toplam (sum(0)) yapıldığı için geriye yayılımda gradyan 
# 32 satır boyunca broadcast edilir ve 1/n ile ölçeklenir.
# Kural 4 branching: İki koldan gelen gradyanlar toplanır (+=).
dhprebn += (1.0 / n) * torch.ones_like(hprebn) * dbnmeani

# ARTIK HER İKİ KOL DA BİRLEŞTİ, TEST ŞİMDİ DOĞRULANABİLİR
cmp('hprebn', dhprebn, hprebn)

hprebn          | exact: True  | approximate: True  | maxdiff: 0.0


#### 18. Doğrusal Katman 1 (`dhprebn` $\to$ `dembcat`, `dW1`, `db1`)

- **İleri Yayılım:** `hprebn = embcat @ W1 + b1`

- `embcat`: `(32, 30)`, `W1`: `(30, 64)`, `b1`: `(64,)`, `hprebn`: `(32, 64)`

- **Mantık & Boyut Eşleme:**
Matris çarpımı türevinde formül ezberlemek yerine **Boyut Kuralı (Kural 1)** uygulanır:

- `dembcat` boyutu `(32, 30)` olmalı $\to$ `dhprebn (32, 64)` ile `W1.T (64, 30)` çarpılmalı: `dhprebn @ W1.T`.

- `dW1` boyutu `(30, 64)` olmalı $\to$ `embcat.T (30, 32)` ile `dhprebn (32, 64)` çarpılmalı: `embcat.T @ dhprebn`.

- `db1` boyutu `(64,)` olmalı $\to$ `b1` satırlar boyunca broadcast edildiği için 0. boyut boyunca toplanmalı: `dhprebn.sum(0)`.

---

In [54]:
# hprebn = embcat @ W1 + b1 # Linear layer 1 # türevi alıncak kod

dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)

cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)

# NOT: 'exact: False' ancak 'approximate: True' çıkması normaldir.
# Donanım seviyesindeki kayan noktalı sayı (float32) yuvarlama farklarından kaynaklanır.

embcat          | exact: True  | approximate: True  | maxdiff: 0.0
W1              | exact: True  | approximate: True  | maxdiff: 0.0
b1              | exact: True  | approximate: True  | maxdiff: 0.0


#### 19. Görünüm Geri Sarma (`dembcat` $\to$ `demb`)

- **İleri Yayılım:** `embcat = emb.view(emb.shape[0], -1)`

(`emb` boyutu `(32, 3, 10)` iken `embcat` boyutu `(32, 30)` yapılmıştı).

- **Mantık & Türev:**
- `view` işlemi bellekteki verileri kopyalamaz; yalnızca tensörün okunma biçimini değiştirir.

- Geriye yayılımda yapılacak işlem, gradyan tensörünü orijinal `emb` boyutuna (`32, 3, 10`) geri döndürmektir (`view`).

- **Nasıl Kurgulanır?**
`demb = dembcat.view(emb.shape)`.

---

In [55]:
# embcat = emb.view(emb.shape[0], -1) # türevi alıncak kod

demb = dembcat.view(emb.shape)

cmp('emb', demb, emb)

emb             | exact: True  | approximate: True  | maxdiff: 0.0


#### 20. Embedding Lookup Tablosu (`demb` $\to$ `dC`)

- **İleri Yayılım:** `emb = C[Xb]`

(Karakter indekslerine göre $C \in \mathbb{R}^{27 \times 10}$ tablosundan satırlar çekilmişti).

- **Mantık & Türev:**
- `Xb` tensöründeki indeksler, `demb` içindeki gradyanların $C$ tablosunun hangi satırına ait olduğunu gösterir.

- Aynı karakter mini-batch içinde birden fazla kez kullanılmış olabileceğinden (**Kural 4**), ilgili satırlara gelen tüm gradyanlar toplanarak (`+=`) aktarılır.

- **Nasıl Kurgulanır?**
$C$ ile aynı boyutta sıfır tensörü açılır (`torch.zeros_like(C)`) ve `Xb` indeksleri taranarak `dC[ix] += demb[k, j]` yapılır.

---

In [56]:
# emb = C[Xb] # embed the characters into vectors # türevi alıncak kod

dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        ix = Xb[k, j]
        dC[ix] += demb[k, j]

cmp('C', dC, C)

C               | exact: True  | approximate: True  | maxdiff: 0.0


### Görev 2 (Egzersiz 2): Cross Entropy Kaybının Analitik Geri Yayılımı

Egzersiz 1'de `loss` noktasından `logits` tensörüne ulaşmak için 8 ayrı ara tensör üzerinden geriye yayılım yaptık (`logprobs`, `probs`, `counts_sum_inv`, `counts_sum`, `counts`, `norm_logits`, `logit_maxes`). 

Ancak kağıt üzerinde analitik türev aldığımızda:
- Tek bir örnek için softmax olasılığı: $P_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$
- Kayıp fonksiyonu: $L = -\ln(P_y)$ (burada $y$ doğru sınıf etiketidir).

Bu ifadenin $z_i$ logit değerine göre türevi alındığında inanılmaz bir sadeleşme olur:
$$\frac{\partial L}{\partial z_i} = \begin{cases} P_i - 1, & \text{eğer } i = y \text{ (doğru sınıf)} \\ P_i, & \text{eğer } i \neq y \text{ (yanlış sınıf)} \end{cases}$$

Yani geriye yayılımda yapmamız gereken tek şey:
1. Logitlerden softmax olasılıklarını ($P$) hesaplamak.
2. Doğru sınıfların olasılığından $1$ çıkarmak ($P - 1$).
3. Mini-batch ortalamasından gelen $\frac{1}{N}$ katsayısı ile ölçeklemek.

In [57]:
# CROSS ENTROPY TEK SATIR ANALİTİK TÜREVİ

# 1. Hızlı İleri Yayılım (F.cross_entropy ile aynı sonucu verdiğini doğrulama)
loss_fast = F.cross_entropy(logits, Yb)
print(f'Hızlı kayıp farkı: {(loss_fast - loss).item()}')

# 2. Analitik dlogits Hesabı:
# Adım a: Logitlerin softmax olasılıklarını alıyoruz (satır bazında)
dlogits = F.softmax(logits, 1)

# Adım b: Doğru etiketlerin (Yb) bulunduğu sütunlardan 1 çıkarıyoruz (P_i - 1)
dlogits[range(n), Yb] -= 1.0

# Adım c: Kayıp hesaplanırken batch ortalaması (1/n) alındığı için gradyanı n'e bölüyoruz
dlogits /= n

# PyTorch'un autograd sonucu ile karşılaştırma:
cmp('logits', dlogits, logits)

Hızlı kayıp farkı: 0.0
logits          | exact: False | approximate: True  | maxdiff: 1.0244548320770264e-08


#### Çıktının Anlamı ve Sezgisel Açıklaması (Push-Pull Kuvveti)

`cmp('logits', dlogits, logits)` çalıştırıldığında `approximate: True` ve `maxdiff: ~1e-9` elde ederiz.

Bu analitik gradyanın sezgisel anlamı muazzamdır:
1. **İtme ve Çekme Kuvveti (Push & Pull):** 
   - Yanlış sınıfların gradyanı $+P_i$'dir; yani model yanlış sınıfa ne kadar yüksek olasılık verdiyse, o kadar güçlü bir **itme** kuvveti uygulanır.
   - Doğru sınıfın gradyanı $P_y - 1$'dir; yani model doğru sınıfa yaklaştıkça bu kuvvet sıfırlanır, uzaksa güçlü bir **çekme** kuvveti uygulanır.
2. **Kuvvetler Dengesi:** 
   - Her bir satırdaki gradyanların toplamı $\sum_i \frac{\partial L}{\partial z_i} = \sum_i P_i - 1 = 1 - 1 = 0$ olur! 
   - Ağın çıkışındaki tüm çekme ve itme kuvvetleri kusursuz bir denge içindedir.

### Görev 3 (Egzersiz 3): BatchNorm Katmanının Analitik Geri Yayılımı

Egzersiz 1'de BatchNorm katmanından geriye doğru geçerken 6 ayrı ara tensör hesapladık (`bnraw`, `bndiff`, `bndiff2`, `bnvar`, `bnvar_inv`, `bnmeani`). 

BatchNorm ileri yayılımı tek bir formül olarak yazılabilir:
$$\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}$$

Karpathy'nin kağıt üzerinde zincir kuralı uygulayarak türettiği gibi, girdi tensörü $x_i$'ye (`hprebn`) giden gradyan; $\mu$ (ortalama), $\sigma^2$ (varyans) ve $\hat{x}_i$ (normalize değer) yollarının birleşmesiyle sadeleşir.

Tüm terimler düzenlenip ortak paranteze alındığında, $\frac{\partial L}{\partial x_i}$ için şu kapalı formül elde edilir[cite: 1]:
$$\frac{\partial L}{\partial x_i} = \frac{\gamma}{\sqrt{\sigma^2 + \epsilon}} \cdot \frac{1}{N} \left[ N \cdot \frac{\partial L}{\partial y_i} - \sum_{j=1}^N \frac{\partial L}{\partial y_j} - \hat{x}_i \sum_{j=1}^N \left( \frac{\partial L}{\partial y_j} \cdot \hat{x}_j \right) \right]$$

Bu formülde:
- $\frac{\partial L}{\partial y_i}$: `dhpreact` (BatchNorm çıkışına gelen gradyan)[cite: 1]
- $\hat{x}_i$: `bnraw` (Normalizasyon sonrası sıfır ortalamalı, birim varyanslı tensör)
- $\gamma$: `bngain` (Öğrenilebilir ölçek parametresi)
- $\sqrt{\sigma^2 + \epsilon}$: `bnvar + 1e-5` teriminin karekökü (yani `1.0 / bnvar_inv`)
- $N$: Batch boyutu (`n = 32`)

In [58]:
# Analitik dhprebn hesabı:
# dhpreact: (32, 64)
# bngain: (1, 64)
# bnraw: (32, 64)
# bnvar_inv: (1, 64)

# Formülün parçalarını tensör seviyesinde bir araya getiriyoruz:
dhprebn = bngain * bnvar_inv / n * (n * dhpreact - dhpreact.sum(0, keepdim=True) - n / (n - 1) * bnraw * (dhpreact * bnraw).sum(0, keepdim=True))

# NOT: n / (n - 1) çarpanı ileri yayılımda Bessel düzeltmesi (1 / (n-1)) 
# kullandığımız için gelir. Standart BatchNorm'da bu çarpan sadece 1'dir.

# PyTorch ve Egzersiz 1 sonucu ile karşılaştırma:
cmp('hprebn', dhprebn, hprebn)

hprebn          | exact: False | approximate: True  | maxdiff: 9.313225746154785e-10


#### Çıktının Anlamı ve Sezgisel Yorumu

`cmp('hprebn', dhprebn, hprebn)` çalıştırıldığında `approximate: True` ve `maxdiff: ~1e-9` elde ederiz

Parantez içindeki 3 terimin fiziksel/istatistiksel anlamı şudur:
1. **$N \cdot \frac{\partial L}{\partial y_i}$:** Ana gradyan sinyali (doğrudan gelen etki)
2. **$-\sum \frac{\partial L}{\partial y_j}$:** Ortalama çıkarmanın ($\mu$) getirdiği düzeltme. Bu terim sayesinde gradyanların sütun bazında toplamı sıfırlanır; yani model girdiyi sabit bir sayıyla kaydırırsa gradyan bunu anında yok sayar
3. **$-\hat{x}_i \sum (\frac{\partial L}{\partial y_j} \cdot \hat{x}_j)$:** Varyans normalizasyonunun ($\sigma^2$) getirdiği düzeltme. Girdinin ölçeğindeki (büyüklüğündeki) değişimleri dengeler

Böylece 6 farklı ara tensörün bellekte tutulmasına gerek kalmadan, doğrudan tek bir CUDA çekirdeğinde türev hesaplanmış olur